# 03. LightGBM 학습 및 예측

## 🎯 목적

02단계에서 생성한 통합 데이터셋으로 **전체 통합 LightGBM 모델** 학습

## 📋 책임

### ✅ 이 단계가 하는 것
1. **Target 생성**: 5일 후 로그 수익률 (target_log_return)
2. **Walk-Forward 분할**: 날짜 기준 train/valid/test
3. **모델 학습**: ticker를 categorical feature로 활용
4. **평가 및 저장**: 성능 지표 + 모델 아티팩트

## 🔄 데이터 흐름

```
data/03_processed/dataset.parquet
    ↓ [load]
  + Target 생성 (종목별 그룹 연산)
    ↓ [walk_forward_split]
  train / valid (4 folds) / test
    ↓ [LightGBM 학습]
  Unified Model (ticker as categorical)
    ↓ [save]
data/04_models/*.pkl
data/05_results/*.csv
```

## 🔧 Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings

from src.models.lightgbm_model import LightGBMModel
from src.modeling.trainer import WalkForwardTrainer
from src.models.artifact import save_model_artifact

warnings.filterwarnings('ignore')
print("✅ Modules loaded")

In [ ]:
# ==================== 설정 ====================

REFERENCE_DATE = "20260112"
DATASET_PATH = Path(f"data/03_processed/dataset_{REFERENCE_DATE}.parquet")
MODEL_DIR = Path("data/04_models")
RESULT_DIR = Path("data/05_results")

# 디렉토리 생성
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Target 설정
TARGET_HORIZON = 10  # 10일 후 예측
TARGET_COL = "target_log_return"

# Walk-Forward 설정
TRAIN_END = "2025-01-13"  # 학습 종료일
VALID_WINDOW = 46         # 검증 구간 (거래일)
TEST_WINDOW = 61          # 테스트 구간
NUM_VALID = 4             # 검증 구간 개수

# LightGBM 하이퍼파라미터
LGBM_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
}

print(f"📁 Dataset: {DATASET_PATH}")
print(f"🎯 Target: {TARGET_HORIZON}일 후 로그 수익률")
print(f"📅 Train end: {TRAIN_END}")
print(f"📊 Valid: {NUM_VALID} folds x {VALID_WINDOW} days")
print(f"🧪 Test: {TEST_WINDOW} days")

## 1️⃣ 데이터 로드 및 Target 생성

In [ ]:
# ==================== 데이터 로드 ====================

print("📥 Loading dataset...")
df = pd.read_parquet(DATASET_PATH)

print(f"   - Shape: {df.shape}")
print(f"   - Tickers: {df['ticker'].nunique():,}")
print(f"   - Period: {df['date'].min()} ~ {df['date'].max()}")

# Feature 컬럼 자동 추출 (ticker 제외)
feature_cols = [c for c in df.columns if c.startswith('feature_')]
print(f"\n✅ Features: {len(feature_cols)}개")
print(f"   Technical: {[c for c in feature_cols if 'tech' in c][:3]}...")
print(f"   Meta: {[c for c in feature_cols if 'meta' in c] or 'None'}")

# Meta Feature 확인
meta_features = [c for c in df.columns if c in ['liquidity_score', 'risk_composite']]
if meta_features:
    print(f"\n💡 Available Meta Features: {meta_features}")
    print("   These will help model learn stock characteristics without ticker ID")
else:
    print("\n⚠️  No meta features found. Consider adding sector/mcap features in 02 stage.")

In [ ]:
# ==================== Target 생성 ====================

print(f"\n🎯 Creating Target: {TARGET_HORIZON}일 후 로그 수익률...")

# 종목별 그룹 연산
df[TARGET_COL] = df.groupby('ticker')['close'].transform(
    lambda x: np.log(x.shift(-TARGET_HORIZON) / x)
)

# NaN 통계
nan_count = df[TARGET_COL].isna().sum()
print(f"   NaN: {nan_count:,} / {len(df):,} ({nan_count/len(df)*100:.1f}%)")
print(f"   → 마지막 {TARGET_HORIZON}일치 데이터는 Target 없음 (정상)")

# Target 분포 확인
print(f"\n📊 Target Distribution:")
print(df[TARGET_COL].describe())

# 극단값 체크
extreme_threshold = 0.5  # 로그 수익률 ±50% 이상
extreme_count = (df[TARGET_COL].abs() > extreme_threshold).sum()
print(f"\n⚠️  Extreme values (|log_return| > {extreme_threshold}): {extreme_count} ({extreme_count/len(df)*100:.3f}%)")

## 2️⃣ 모델 학습 (Walk-Forward)

In [ ]:
# ==================== 모델 초기화 ====================

print("🔧 Initializing LightGBM Model...")

# Ticker는 Feature에서 제외 (신규 종목 예측 가능하도록)
# Meta features (liquidity, risk 등)가 종목 특성 대체

model = LightGBMModel(
    model_version="v1_unified_no_ticker",
    params=LGBM_PARAMS,
    feature_list=feature_cols,  # ticker 제외됨
    categorical_features=[],     # 카테고리 feature 없음
    task="regression"
)

print(f"   Model: {model.model_name} v{model.model_version}")
print(f"   Features: {len(feature_cols)} (NO ticker - enables prediction for new stocks)")
print(f"   Hyperparameters: {model.params}")

In [ ]:
# ==================== Trainer 실행 ====================

print("\n" + "="*65)
print("🚀 Starting Walk-Forward Training...")
print("="*65)

trainer = WalkForwardTrainer(
    model=model,
    feature_cols=feature_cols,  # ticker 제외됨
    target_col=TARGET_COL,
    date_col='date',
    categorical_features=[]  # 빈 리스트
)

results = trainer.run(
    df=df,
    train_end=TRAIN_END,
    valid_window_days=VALID_WINDOW,
    test_window_days=TEST_WINDOW,
    num_valid=NUM_VALID,
    fit_kwargs={
        'num_boost_round': 1000,
        'callbacks': [
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    }
)

print("\n✅ Training completed!")

## 3️⃣ 성능 평가

In [ ]:
# ==================== 성능 지표 출력 ====================

print("\n" + "="*65)
print("📊 Performance Summary")
print("="*65)

# Train
print("\n[Train Set]")
for k, v in results['train_metrics'].items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# Valid (평균)
print("\n[Validation Set - Average]")
valid_avg = {}
for key in results['valid_metrics'][0].keys():
    if key != 'samples':
        valid_avg[key] = np.mean([v[key] for v in results['valid_metrics']])
    else:
        valid_avg[key] = sum([v[key] for v in results['valid_metrics']])

for k, v in valid_avg.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# Test
print("\n[Test Set]")
for k, v in results['test_metrics'].items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

# 과적합 지표
print("\n[Overfitting Check]")
rmse_gap = results['test_metrics']['rmse'] - valid_avg['rmse']
print(f"  RMSE gap (test - valid): {rmse_gap:+.6f}")
if rmse_gap > 0.02:
    print("  ⚠️  과적합 의심")
else:
    print("  ✅ 일반화 양호")

In [ ]:
# ==================== Feature Importance ====================

print("\n📈 Feature Importance (Top 10):")

meta = model.get_meta()
importance_dict = meta.get('feature_importance', {})

if importance_dict:
    importance_df = pd.DataFrame([
        {'feature': k, 'importance': v} 
        for k, v in importance_dict.items()
    ]).sort_values('importance', ascending=False)
    
    print(importance_df.head(10).to_string(index=False))
    
    # 저장
    importance_path = RESULT_DIR / f"feature_importance_{REFERENCE_DATE}.csv"
    importance_df.to_csv(importance_path, index=False)
    print(f"\n💾 Saved: {importance_path}")
else:
    print("  (Feature importance not available)")

## 4️⃣ 모델 및 결과 저장

In [ ]:
# ==================== 모델 저장 ====================

print("\n💾 Saving model artifact...")

artifact_path = save_model_artifact(
    model_name="lightgbm",
    model_version="v1_unified_no_ticker",
    model_object=model,
    metadata={
        "feature_list": feature_cols,
        "training_period": f"~{TRAIN_END}",
        "hyperparameters": LGBM_PARAMS,
        "data_version": REFERENCE_DATE,
        "target_horizon": TARGET_HORIZON,
        "test_rmse": results['test_metrics']['rmse'],
        "test_r2": results['test_metrics']['r2'],
        "note": "No ticker feature - can predict new stocks",
    },
    model_dir=MODEL_DIR
)

print(f"✅ Model saved: {artifact_path}")
print(f"\n💡 This model can predict NEW stocks (신규 상장 종목 예측 가능)")

In [ ]:
# ==================== 예측 결과 저장 ====================

print("\n💾 Saving predictions...")

pred_df = results['test_predictions']
pred_path = RESULT_DIR / f"predictions_{REFERENCE_DATE}.csv"
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

print(f"✅ Predictions saved: {pred_path}")
print(f"   Shape: {pred_df.shape}")
print(f"\n📋 Sample:")
print(pred_df.head(10))

In [ ]:
# ==================== 지표 저장 ====================

print("\n💾 Saving metrics summary...")

metrics_summary = {
    'reference_date': REFERENCE_DATE,
    'train_end': TRAIN_END,
    'target_horizon': TARGET_HORIZON,
    **{f'train_{k}': v for k, v in results['train_metrics'].items()},
    **{f'valid_avg_{k}': v for k, v in valid_avg.items()},
    **{f'test_{k}': v for k, v in results['test_metrics'].items()},
}

metrics_df = pd.DataFrame([metrics_summary])
metrics_path = RESULT_DIR / f"metrics_summary_{REFERENCE_DATE}.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"✅ Metrics saved: {metrics_path}")

## ✅ 완료!

### 생성된 파일
- 모델: `data/04_models/lightgbm/*.pkl`
- 예측: `data/05_results/predictions_*.csv`
- 지표: `data/05_results/metrics_summary_*.csv`
- Feature Importance: `data/05_results/feature_importance_*.csv`

### 다음 단계
- **트랙 D**: Universe 선정 및 시그널 생성
- **백테스트**: 실제 매매 시뮬레이션

### 모델 로드 예시
```python
from src.models.lightgbm_model import LightGBMModel

loaded_model = LightGBMModel.load('data/04_models/lightgbm/*.pkl')
predictions = loaded_model.predict(new_data)
```